In [1]:
import os, sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"  # JDK 17 (pyspark 4.x exige Java 17+)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"  # winutils.exe/hadoop.dll para o Spark funcionar com FS local no Windows
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

import os
os.environ['SPARK_LOCAL_IP'] = '192.168.15.16'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, greatest, lit
from pyspark.sql.functions import col, sum, when
from pyspark.sql.functions import col, when, sum as _sum, avg, max as _max, min as _min, stddev, percentile_approx
from pyspark.sql import Window
from pyspark.sql.functions import lag


spark = SparkSession \
    .builder \
    .appName("HomeCredit_Credit_Card_Balance") \
    .master("local[1]") \
    .config("spark.driver.host", "192.168.15.16") \
    .config("spark.driver.bindAddress", "192.168.15.16") \
    .getOrCreate()

print(spark.version)

c:\Users\muril\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [2]:
# Cell 1 original (carrega só o credit_card_balance.csv):
file_path_credit_card_balance = r"C:\Users\muril\OneDrive\Desktop\Fonte\credit_card_balance.csv"
dados = spark.read.csv(file_path_credit_card_balance, header=True, inferSchema=True)

dados.createOrReplaceTempView("dados")
dados.show(5)

+----------+----------+--------------+-----------+-----------------------+------------------------+--------------------+--------------------------+------------------------+-----------------------+-------------------+-------------------------+------------------------+-------------+--------------------+------------------------+--------------------+--------------------------+------------------------+-------------------------+--------------------+------+----------+
|SK_ID_PREV|SK_ID_CURR|MONTHS_BALANCE|AMT_BALANCE|AMT_CREDIT_LIMIT_ACTUAL|AMT_DRAWINGS_ATM_CURRENT|AMT_DRAWINGS_CURRENT|AMT_DRAWINGS_OTHER_CURRENT|AMT_DRAWINGS_POS_CURRENT|AMT_INST_MIN_REGULARITY|AMT_PAYMENT_CURRENT|AMT_PAYMENT_TOTAL_CURRENT|AMT_RECEIVABLE_PRINCIPAL|AMT_RECIVABLE|AMT_TOTAL_RECEIVABLE|CNT_DRAWINGS_ATM_CURRENT|CNT_DRAWINGS_CURRENT|CNT_DRAWINGS_OTHER_CURRENT|CNT_DRAWINGS_POS_CURRENT|CNT_INSTALMENT_MATURE_CUM|NAME_CONTRACT_STATUS|SK_DPD|SK_DPD_DEF|
+----------+----------+--------------+-----------+------------------

Colunas de razões derivadas + antigas

In [3]:
dados = spark.sql("""

    select
        *,
        -- Mantidas
        AMT_BALANCE / nullif(AMT_CREDIT_LIMIT_ACTUAL, 0) as RATIO_UTILIZACAO_LIMITE,
        AMT_PAYMENT_TOTAL_CURRENT / nullif(AMT_INST_MIN_REGULARITY, 0) as RATIO_PAGAMENTO_MINIMO,
        AMT_PAYMENT_TOTAL_CURRENT / nullif(AMT_BALANCE, 0) as RATIO_PAGAMENTO_SALDO,
        AMT_DRAWINGS_ATM_CURRENT / nullif(AMT_DRAWINGS_CURRENT, 0) as RATIO_SAQUE_ATM_SOBRE_TOTAL,
        AMT_DRAWINGS_CURRENT / nullif(AMT_CREDIT_LIMIT_ACTUAL, 0) as RATIO_SAQUE_SOBRE_LIMITE,

        -- NOVAS
        -- Proporção de juros/encargos sobre o total a receber — quanto maior, mais "cara" a dívida do cliente
        (AMT_TOTAL_RECEIVABLE - AMT_RECEIVABLE_PRINCIPAL) / nullif(AMT_TOTAL_RECEIVABLE, 0) as RATIO_JUROS_SOBRE_RECEBIVEL,
        -- Proporção do principal sobre o total (inverso do acima, mantém como variável própria)
        AMT_RECEIVABLE_PRINCIPAL / nullif(AMT_TOTAL_RECEIVABLE, 0) as RATIO_PRINCIPAL_SOBRE_RECEBIVEL,
        -- Frequência de saque em ATM (contagem, não valor) — comportamento, não volume
        CNT_DRAWINGS_ATM_CURRENT / nullif(CNT_DRAWINGS_CURRENT, 0) as RATIO_QTD_SAQUE_ATM_SOBRE_TOTAL,
        -- Valor de saque via POS (compras) sobre o total sacado
        AMT_DRAWINGS_POS_CURRENT / nullif(AMT_DRAWINGS_CURRENT, 0) as RATIO_SAQUE_POS_SOBRE_TOTAL,
        -- Pagamento da parcela do mês vs pagamento total feito no mês (parcial vs quitação maior)
        AMT_PAYMENT_CURRENT / nullif(AMT_PAYMENT_TOTAL_CURRENT, 0) as RATIO_PAGAMENTO_ATUAL_SOBRE_TOTAL
    from dados

""")

dados.createOrReplaceTempView("dados")
dados.show(5)

+----------+----------+--------------+-----------+-----------------------+------------------------+--------------------+--------------------------+------------------------+-----------------------+-------------------+-------------------------+------------------------+-------------+--------------------+------------------------+--------------------+--------------------------+------------------------+-------------------------+--------------------+------+----------+-----------------------+----------------------+---------------------+---------------------------+------------------------+---------------------------+-------------------------------+-------------------------------+---------------------------+---------------------------------+
|SK_ID_PREV|SK_ID_CURR|MONTHS_BALANCE|AMT_BALANCE|AMT_CREDIT_LIMIT_ACTUAL|AMT_DRAWINGS_ATM_CURRENT|AMT_DRAWINGS_CURRENT|AMT_DRAWINGS_OTHER_CURRENT|AMT_DRAWINGS_POS_CURRENT|AMT_INST_MIN_REGULARITY|AMT_PAYMENT_CURRENT|AMT_PAYMENT_TOTAL_CURRENT|AMT_RECEIVABLE_P

Criando as flags de janelas temporais

In [4]:
dados = spark.sql("""

    select
        *,
        case when MONTHS_BALANCE >= -3   then 1 else 0 end as flag_ultimos_3_meses,
        case when MONTHS_BALANCE >= -6  then 1 else 0 end as flag_ultimos_6_meses,
        case when MONTHS_BALANCE >= -9  then 1 else 0 end as flag_ultimos_9_meses,
        case when MONTHS_BALANCE >= -12  then 1 else 0 end as flag_ultimos_12_meses,
        case when MONTHS_BALANCE >= -18  then 1 else 0 end as flag_ultimos_18_meses,
        case when MONTHS_BALANCE >= -24  then 1 else 0 end as flag_ultimos_24_meses,
        case when MONTHS_BALANCE >= -36  then 1 else 0 end as flag_ultimos_36_meses
    from dados

""")

dados.createOrReplaceTempView("dados")

colunas_flags = ['flag_ultimos_3_meses', 'flag_ultimos_6_meses', 'flag_ultimos_9_meses',
                  'flag_ultimos_12_meses', 'flag_ultimos_18_meses', 'flag_ultimos_24_meses',
                  'flag_ultimos_36_meses']

#### Coluna categórica

In [5]:
status_possiveis_cc = [row["NAME_CONTRACT_STATUS"] for row in dados.select("NAME_CONTRACT_STATUS").distinct().collect() if row["NAME_CONTRACT_STATUS"] is not None]
print(status_possiveis_cc)

for status in status_possiveis_cc:
    status_slug = status.replace(' ', '_').upper()
    dados = dados.withColumn(f"flag_status_{status_slug}", when(col("NAME_CONTRACT_STATUS") == status, 1).otherwise(0))

dados.createOrReplaceTempView("dados")

['Demand', 'Approved', 'Completed', 'Active', 'Signed', 'Sent proposal', 'Refused']


#### Flags faixa de atraso

In [6]:
dados = spark.sql("""

    select
        *,
        case when SK_DPD = 0 then 1 else 0 end as flag_atraso_em_dia,
        case when SK_DPD between 1 and 15 then 1 else 0 end as flag_atraso_1_15,
        case when SK_DPD between 16 and 30 then 1 else 0 end as flag_atraso_16_30,
        case when SK_DPD between 31 and 60 then 1 else 0 end as flag_atraso_31_60,
        case when SK_DPD between 61 and 90 then 1 else 0 end as flag_atraso_61_90,
        case when SK_DPD > 90 then 1 else 0 end as flag_atraso_90_mais
    from dados

""")

dados.createOrReplaceTempView("dados")

#### Marcação de medidas estatíticas fora das janelas temporais

In [7]:
colunas_numericas_cc = [
    # Mantidas
    'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_ATM_CURRENT',
    'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_INST_MIN_REGULARITY', 'SK_DPD', 'SK_DPD_DEF',
    'RATIO_UTILIZACAO_LIMITE', 'RATIO_PAGAMENTO_MINIMO', 'RATIO_PAGAMENTO_SALDO',
    'RATIO_SAQUE_ATM_SOBRE_TOTAL', 'RATIO_SAQUE_SOBRE_LIMITE',

    # NOVAS — colunas brutas ainda não usadas
    'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_PAYMENT_CURRENT',
    'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
    'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT',
    'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM',

    # NOVAS — ratios criados no Bloco 1
    'RATIO_JUROS_SOBRE_RECEBIVEL', 'RATIO_PRINCIPAL_SOBRE_RECEBIVEL',
    'RATIO_QTD_SAQUE_ATM_SOBRE_TOTAL', 'RATIO_SAQUE_POS_SOBRE_TOTAL', 'RATIO_PAGAMENTO_ATUAL_SOBRE_TOTAL'
]

In [8]:
expressoes_gerais = []

for coluna in colunas_numericas_cc:
    expressoes_gerais.append(avg(col(coluna)).alias(f"MEAN_{coluna}"))
    expressoes_gerais.append(percentile_approx(col(coluna), 0.5).alias(f"MEDIAN_{coluna}"))
    expressoes_gerais.append(_max(col(coluna)).alias(f"MAX_{coluna}"))
    expressoes_gerais.append(_min(col(coluna)).alias(f"MIN_{coluna}"))
    expressoes_gerais.append(stddev(col(coluna)).alias(f"STD_{coluna}"))

expressoes_gerais.append(_sum(lit(1)).alias("QTD_TOTAL_MESES_CC"))

expressoes_gerais = tuple(expressoes_gerais)

book_cc_geral = dados.groupBy("SK_ID_CURR").agg(*expressoes_gerais).orderBy("SK_ID_CURR")

print((book_cc_geral.count(), len(book_cc_geral.columns)))
book_cc_geral.createOrReplaceTempView("df_temp01")
book_cc_geral.show(5)

(103558, 147)
+----------+-----------------+------------------+---------------+---------------+------------------+----------------------------+------------------------------+---------------------------+---------------------------+---------------------------+-------------------------+---------------------------+------------------------+------------------------+------------------------+-----------------------------+-------------------------------+----------------------------+----------------------------+----------------------------+------------------------------+--------------------------------+-----------------------------+-----------------------------+-----------------------------+----------------------------+------------------------------+---------------------------+---------------------------+---------------------------+--------------------+-------------+----------+----------+-------------------+--------------------+-----------------+--------------+--------------+-------------------+

#### Medidas Estatísticas dentro das janelas temporais

In [9]:
expressoes_temporal = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_numericas_cc:
        base = when(col(flag) == 1, col(coluna))

        expressoes_temporal.append(avg(base).alias(f"MEAN_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(percentile_approx(base, 0.5).alias(f"MEDIAN_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(_max(base).alias(f"MAX_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(_min(base).alias(f"MIN_{coluna}_{sufixo_janela}"))
        expressoes_temporal.append(stddev(base).alias(f"STD_{coluna}_{sufixo_janela}"))

expressoes_temporal = tuple(expressoes_temporal)

book_cc_temporal = dados.groupBy("SK_ID_CURR").agg(*expressoes_temporal).orderBy("SK_ID_CURR")

print((book_cc_temporal.count(), len(book_cc_temporal.columns)))
book_cc_temporal.createOrReplaceTempView("df_temp02")
book_cc_temporal.show(5)

(103558, 1016)
+----------+-------------------+---------------------+------------------+------------------+------------------+-------------------------------+---------------------------------+------------------------------+------------------------------+------------------------------+----------------------------+------------------------------+---------------------------+---------------------------+---------------------------+--------------------------------+----------------------------------+-------------------------------+-------------------------------+-------------------------------+---------------------------------+-----------------------------------+--------------------------------+--------------------------------+--------------------------------+-------------------------------+---------------------------------+------------------------------+------------------------------+------------------------------+--------------+----------------+-------------+-------------+-------------+-----

#### Razões entre as janelas

In [10]:
janelas_ordem = ['U3', 'U6', 'U9', 'U12', 'U18', 'U24', 'U36']
pares_u3_vs_demais = [(janelas_ordem[0], j) for j in janelas_ordem[1:]]
pares_consecutivos = [(janelas_ordem[i], janelas_ordem[i + 1]) for i in range(1, len(janelas_ordem) - 1)]

def gerar_expressoes_razao(df, pares, colunas, metrica='MEAN'):
    colunas_existentes = set(df.columns)
    expressoes = []
    for numerador, denominador in pares:
        for coluna in colunas:
            col_num = f"{metrica}_{coluna}_{numerador}"
            col_den = f"{metrica}_{coluna}_{denominador}"
            if col_num in colunas_existentes and col_den in colunas_existentes:
                expressoes.append(
                    (col(col_num) / when(col(col_den) != 0, col(col_den)))
                    .alias(f"RATIO_{coluna}_{metrica}_{numerador}_{denominador}")
                )
    return expressoes

expressoes_razao_a = gerar_expressoes_razao(book_cc_temporal, pares_u3_vs_demais, colunas_numericas_cc, metrica='MEAN')
expressoes_razao_b = gerar_expressoes_razao(book_cc_temporal, pares_consecutivos, colunas_numericas_cc, metrica='MEAN')

book_cc_razoes = book_cc_temporal.select("SK_ID_CURR", *expressoes_razao_a, *expressoes_razao_b)

print((book_cc_razoes.count(), len(book_cc_razoes.columns)))
book_cc_razoes.createOrReplaceTempView("df_temp03")
book_cc_razoes.show(5)

(103558, 320)
+----------+----------------------------+----------------------------------------+-------------------------------------+-----------------------------------------+------------------------------------------+----------------------------------------+-----------------------+---------------------------+----------------------------------------+---------------------------------------+--------------------------------------+--------------------------------------------+-----------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------+-----------------------------------------+------------------------------+-------------------------------------+-----------------------------------------+-------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------+--------------------------------------

#### Range e CV por janelas

In [11]:
expressoes_range_cv = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_numericas_cc:
        base = when(col(flag) == 1, col(coluna))
        expressoes_range_cv.append((_max(base) - _min(base)).alias(f"RANGE_{coluna}_{sufixo_janela}"))
        expressoes_range_cv.append((stddev(base) / when(avg(base) != 0, avg(base))).alias(f"CV_{coluna}_{sufixo_janela}"))

expressoes_range_cv = tuple(expressoes_range_cv)

book_cc_range_cv = dados.groupBy("SK_ID_CURR").agg(*expressoes_range_cv).orderBy("SK_ID_CURR")

print((book_cc_range_cv.count(), len(book_cc_range_cv.columns)))
book_cc_range_cv.createOrReplaceTempView("df_temp04")
book_cc_range_cv.show(5)

(103558, 407)
+----------+--------------------+-----------------+--------------------------------+-----------------------------+-----------------------------+--------------------------+---------------------------------+------------------------------+----------------------------------+-------------------------------+--------------------------------+-----------------------------+---------------+------------+-------------------+----------------+--------------------------------+-----------------------------+-------------------------------+----------------------------+------------------------------+---------------------------+------------------------------------+---------------------------------+---------------------------------+------------------------------+-----------------------------------+--------------------------------+---------------------------------+------------------------------+----------------------------+-------------------------+---------------------------------+------------

#### Contagem por Status

In [12]:
colunas_status_slug = [f"flag_status_{s.replace(' ', '_').upper()}" for s in status_possiveis_cc]

expressoes_status_janela = []

for flag in colunas_flags:
    for coluna in colunas_status_slug:
        expressoes_status_janela.append(
            _sum(when(col(flag) == 1, col(coluna)).otherwise(0)).alias(f"QTD_{coluna.upper()}_{flag.upper()}")
        )

expressoes_status_janela = tuple(expressoes_status_janela)

book_cc_status_janela = dados.groupBy("SK_ID_CURR").agg(*expressoes_status_janela).orderBy("SK_ID_CURR")

print((book_cc_status_janela.count(), len(book_cc_status_janela.columns)))
book_cc_status_janela.createOrReplaceTempView("df_temp05")
book_cc_status_janela.show(5)

(103558, 50)
+----------+-------------------------------------------+---------------------------------------------+----------------------------------------------+-------------------------------------------+-------------------------------------------+--------------------------------------------------+--------------------------------------------+-------------------------------------------+---------------------------------------------+----------------------------------------------+-------------------------------------------+-------------------------------------------+--------------------------------------------------+--------------------------------------------+-------------------------------------------+---------------------------------------------+----------------------------------------------+-------------------------------------------+-------------------------------------------+--------------------------------------------------+--------------------------------------------+------------

#### Contagem de faixas de atraso por janela

In [13]:
colunas_atraso_flags = ['flag_atraso_em_dia', 'flag_atraso_1_15', 'flag_atraso_16_30',
                          'flag_atraso_31_60', 'flag_atraso_61_90', 'flag_atraso_90_mais']

expressoes_atraso_janela = []

for flag in colunas_flags:
    for coluna in colunas_atraso_flags:
        expressoes_atraso_janela.append(
            _sum(when(col(flag) == 1, col(coluna)).otherwise(0)).alias(f"QTD_{coluna.upper()}_{flag.upper()}")
        )

expressoes_atraso_janela = tuple(expressoes_atraso_janela)

book_cc_atraso_janela = dados.groupBy("SK_ID_CURR").agg(*expressoes_atraso_janela).orderBy("SK_ID_CURR")

print((book_cc_atraso_janela.count(), len(book_cc_atraso_janela.columns)))
book_cc_atraso_janela.createOrReplaceTempView("df_temp06")
book_cc_atraso_janela.show(5)

(103558, 43)
+----------+-------------------------------------------+-----------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+--------------------------------------------+------------------------------------------+-------------------------------------------+-------------------------------------------+-------------------

#### Perfil por cartão (SK_ID_PREV) mantido + expandido com maturidade

In [14]:
book_cc_por_cartao = dados.groupBy("SK_ID_PREV", "SK_ID_CURR").agg(
    _max("SK_DPD").alias("MAX_DPD_CARTAO"),
    avg("SK_DPD").alias("MEAN_DPD_CARTAO"),
    _max("RATIO_UTILIZACAO_LIMITE").alias("MAX_UTILIZACAO_CARTAO"),
    _sum(lit(1)).alias("QTD_MESES_CARTAO"),
    # NOVO — maturidade do cartão (quantas parcelas já foram pagas nesse cartão)
    _max("CNT_INSTALMENT_MATURE_CUM").alias("QTD_PARCELAS_MATURADAS_CARTAO"),
    # NOVO — pior proporção de juros sobre recebível naquele cartão
    _max("RATIO_JUROS_SOBRE_RECEBIVEL").alias("MAX_JUROS_CARTAO")
)

book_cc_perfil_cartoes = book_cc_por_cartao.groupBy("SK_ID_CURR").agg(
    _sum(lit(1)).alias("QTD_CARTOES_ANTERIORES"),
    avg("QTD_MESES_CARTAO").alias("MEAN_DURACAO_CARTOES"),
    _max("MAX_DPD_CARTAO").alias("PIOR_DPD_ENTRE_CARTOES"),
    _max("MAX_UTILIZACAO_CARTAO").alias("PIOR_UTILIZACAO_ENTRE_CARTOES"),
    # NOVO
    avg("QTD_PARCELAS_MATURADAS_CARTAO").alias("MEAN_MATURIDADE_CARTOES"),
    _max("MAX_JUROS_CARTAO").alias("PIOR_JUROS_ENTRE_CARTOES")
)

print((book_cc_perfil_cartoes.count(), len(book_cc_perfil_cartoes.columns)))
book_cc_perfil_cartoes.createOrReplaceTempView("df_temp07")
book_cc_perfil_cartoes.show(5)

(103558, 7)
+----------+----------------------+--------------------+----------------------+-----------------------------+-----------------------+------------------------+
|SK_ID_CURR|QTD_CARTOES_ANTERIORES|MEAN_DURACAO_CARTOES|PIOR_DPD_ENTRE_CARTOES|PIOR_UTILIZACAO_ENTRE_CARTOES|MEAN_MATURIDADE_CARTOES|PIOR_JUROS_ENTRE_CARTOES|
+----------+----------------------+--------------------+----------------------+-----------------------------+-----------------------+------------------------+
|    212175|                     1|                28.0|                     0|                          0.0|                    0.0|                    NULL|
|    357089|                     1|                 5.0|                     0|         0.014510736842105264|                    1.0|                     0.0|
|    206351|                     1|                84.0|                     0|           1.0459093333333334|                    7.0|                     1.0|
|    328279|                     1

## Join Final

In [15]:
book_credit_card = spark.sql("""

    select
        d1.SK_ID_CURR,
        d1.*EXCEPT (SK_ID_CURR),
        d2.*EXCEPT (SK_ID_CURR),
        d3.*EXCEPT (SK_ID_CURR),
        d4.*EXCEPT (SK_ID_CURR),
        d5.*EXCEPT (SK_ID_CURR),
        d6.*EXCEPT (SK_ID_CURR),
        d7.*EXCEPT (SK_ID_CURR)
    from df_temp01 as d1
    left join df_temp02 as d2 on d1.SK_ID_CURR = d2.SK_ID_CURR
    left join df_temp03 as d3 on d1.SK_ID_CURR = d3.SK_ID_CURR
    left join df_temp04 as d4 on d1.SK_ID_CURR = d4.SK_ID_CURR
    left join df_temp05 as d5 on d1.SK_ID_CURR = d5.SK_ID_CURR
    left join df_temp06 as d6 on d1.SK_ID_CURR = d6.SK_ID_CURR
    left join df_temp07 as d7 on d1.SK_ID_CURR = d7.SK_ID_CURR

""")

print((book_credit_card.count(), len(book_credit_card.columns)))

colunas = book_credit_card.columns
duplicadas = [c for c in colunas if colunas.count(c) > 1]
print(set(duplicadas))

(103558, 1984)
set()


In [16]:
df_temp_cc = book_credit_card.repartition(1)
df_temp_cc.write.mode("overwrite").parquet("credit_card_balance_agg.parquet")